# Per-feature configuration and fitted widths

Configure numerical, Boolean and categorical columns, inspect actual fitted widths, and restore the preprocessing state from JSON. The same code appears in the [pipeline tutorial](../docs/tutorials/sklearn_pipeline.md#configure-mixed-columns-with-preprocessor).

## Training data

Keep column dtypes in a DataFrame. The three floating-point columns are numerical; `active` and `city` are categorical. The chosen RBF placement does not use the target.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

from pretab import Preprocessor, RepresentationConfig

frame = pd.DataFrame({
    "age": np.linspace(18.0, 70.0, 80),
    "income": np.linspace(25000.0, 90000.0, 80),
    "tenure": np.linspace(0.0, 30.0, 80),
    "active": [True, False] * 40,
    "city": ["Oslo", "Lima", "Perth", "Oslo"] * 20,
})
target = 0.4 * frame["age"] + 2.0 * frame["active"]

## Request local settings

Mix method strings, mappings and immutable `RepresentationConfig` values. RBF widths are local to each numerical column. The Boolean uses integer encoding; the city encoder drops its first learned category. The deliberately reversed override map does not set output order.

`get_params()` reads the constructor request, including the global width of seven.

In [2]:
pre = Preprocessor(
    numerical_method="none",
    output_dim=7,
    target_aware=False,
    placement_strategy="uniform",
    scaling="none",
    output_format="dense",
    random_state=17,
    feature_preprocessing={
        "city": {"method": "one-hot", "params": {"drop": "first"}},
        "tenure": RepresentationConfig(method="rbf", output_dim=5),
        "active": "int",
        "income": {"method": "rbf", "output_dim": 3, "params": {"gamma": 0.25}},
        "age": RepresentationConfig(method="RBF", output_dim=2),
    },
)
requested = pre.get_params(deep=False)
assert requested["output_dim"] == 7
assert requested["feature_preprocessing"]["age"].output_dim == 2

## Read the resolved configuration

`get_resolved_config()` applies precedence and normalizes aliases without fitting. It reports settings, not learned categories or realized output widths. The global width stays seven while the local age width is two.

In [3]:
resolved = pre.get_resolved_config()
assert resolved["output_dim"] == 7
assert resolved["feature_preprocessing"]["age"]["method"] == "rbf"
assert resolved["feature_preprocessing"]["age"]["output_dim"] == 2
assert resolved["feature_preprocessing"]["income"]["params"]["gamma"] == 0.25
assert not hasattr(pre, "output_dims_")

## Inspect fitted state and actual widths

Fit inside the pipeline so training and prediction use the same preprocessing. The fitted widths are:

| Column | Width |
| --- | ---: |
| `age` | 2 |
| `income` | 3 |
| `tenure` | 5 |
| `active` | 1 |
| `city` | 2 |

The city width depends on its three learned categories and `drop="first"`. Numerical outputs precede categorical outputs, in source order within each kind. Use fitted dimensions to size downstream inputs. Changing a request after fitting does not refit the learned transformers.

In [6]:
mixed_model = Pipeline([("pre", pre), ("ridge", Ridge(alpha=1.0))])
mixed_model.fit(frame, target)

assert pre.numerical_features_ == ["age", "income", "tenure"]
assert pre.categorical_features_ == ["active", "city"]
expected_dims = {"age": 2, "income": 3, "tenure": 5, "active": 1, "city": 2}
assert pre.output_dims_ == expected_dims
assert list(pre.output_dims_) == list(expected_dims)
assert pre.total_output_dim_ == 13

numerical, categorical, embeddings = pre.get_feature_info(verbose=False)
assert {name: info["dimension"] for name, info in numerical.items()} == {
    "age": 2, "income": 3, "tenure": 5,
}
assert {name: info["dimension"] for name, info in categorical.items()} == {
    "active": 1, "city": 2,
}
assert embeddings == {}
matrix = pre.transform(frame, return_array=True)
assert isinstance(matrix, np.ndarray)
assert matrix.shape == (80, 13)
assert pre.get_feature_names_out().shape == (13,)
predictions = mixed_model.predict(frame)
assert isinstance(predictions, np.ndarray)
assert predictions.shape == (80,)

## Save and restore fitted preprocessing

`to_spec()` captures fitted preprocessing state, including typed local settings. Round-trip it through strict JSON, then use `from_spec()` without fitting again. This saves the preprocessing step, not the Ridge model.

For persistence and artifact trust details, see the [reproducibility guide](../docs/core_concepts/reproducibility.md).

In [5]:
spec = json.loads(json.dumps(pre.to_spec(), allow_nan=False))
replayed = Preprocessor.from_spec(spec)

assert replayed.output_dims_ == expected_dims
assert replayed.total_output_dim_ == 13
np.testing.assert_array_equal(replayed.get_feature_names_out(), pre.get_feature_names_out())
np.testing.assert_array_equal(replayed.transform(frame, return_array=True), matrix)
assert replayed.fingerprint_ == pre.fingerprint_